In [5]:
import tkinter as tk
from tkinter import messagebox
import hashlib
import base64
import random
import string
from cryptography.fernet import Fernet


passwords = {}
encryption = None


def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()


def create_key(password):
    key = hashlib.sha256(password.encode()).digest()
    key = base64.urlsafe_b64encode(key)
    return Fernet(key)


def create_master_password():
    master = password_entry.get()
    confirm = confirm_entry.get()

    if not master or not confirm:
        messagebox.showerror("Error", "Please enter both passwords!")
        return

    if master != confirm:
        messagebox.showerror("Error", "Passwords do not match!")
        return

    if len(master) < 6:
        messagebox.showerror(
            "Error",
            "Master password must be at least 6 characters!"
        )
        return

    with open("master.txt", "w") as file:
        file.write(hash_password(master))

    global encryption
    encryption = create_key(master)

    messagebox.showinfo(
        "Success",
        "Master password created successfully!"
    )

    login_window.destroy()
    open_dashboard()


def login():
    master = password_entry.get()

    try:
        with open("master.txt", "r") as file:
            saved_hash = file.read().strip()
    except FileNotFoundError:
        create_master_window()
        return

    if hash_password(master) == saved_hash:
        global encryption
        encryption = create_key(master)

        load_passwords()

        messagebox.showinfo("Success", "Login successful!")

        login_window.destroy()
        open_dashboard()
    else:
        messagebox.showerror("Error", "Wrong master password!")


def load_passwords():
    passwords.clear()

    try:
        with open("password.txt", "r") as file:
            for line in file:
                line = line.strip()

                if not line:
                    continue

                website, encrypted_password = line.split(":", 1)

                password = encryption.decrypt(
                    encrypted_password.encode()
                ).decode()

                passwords[website] = password

    except FileNotFoundError:
        pass
    except Exception:
        messagebox.showerror(
            "Error",
            "Could not decrypt password file!"
        )


def rewrite_password_file():
    with open("password.txt", "w") as file:
        for website, password in passwords.items():
            encrypted_password = encryption.encrypt(
                password.encode()
            ).decode()

            file.write(
                f"{website}:{encrypted_password}\n"
            )


def check_password_strength(password):
    score = 0

    if len(password) >= 8:
        score += 1

    if any(char.isupper() for char in password):
        score += 1

    if any(char.islower() for char in password):
        score += 1

    if any(char.isdigit() for char in password):
        score += 1

    if any(char in "@#$%^&*" for char in password):
        score += 1

    if score <= 2:
        return "Weak"
    elif score <= 4:
        return "Medium"
    else:
        return "Strong"


def save_password():
    website = website_entry.get().strip()
    password = new_password_entry.get()

    if not website or not password:
        messagebox.showerror(
            "Error",
            "Please enter website and password!"
        )
        return

    strength = check_password_strength(password)

    passwords[website] = password
    rewrite_password_file()

    messagebox.showinfo(
        "Saved",
        f"Password saved successfully!\n\nStrength: {strength}"
    )

    clear_fields()


def search_password():
    website = search_entry.get().strip()

    if website in passwords:
        result_label.config(
            text=f"Password: {passwords[website]}"
        )
    else:
        result_label.config(
            text="Website not found!"
        )


def update_password():
    website = website_entry.get().strip()
    new_password = new_password_entry.get()

    if website not in passwords:
        messagebox.showerror(
            "Error",
            "Website not found!"
        )
        return

    if not new_password:
        messagebox.showerror(
            "Error",
            "Enter a new password!"
        )
        return

    passwords[website] = new_password
    rewrite_password_file()

    strength = check_password_strength(new_password)

    messagebox.showinfo(
        "Updated",
        f"Password updated successfully!\n\nStrength: {strength}"
    )

    clear_fields()


def delete_password():
    website = website_entry.get().strip()

    if website not in passwords:
        messagebox.showerror(
            "Error",
            "Website not found!"
        )
        return

    confirm = messagebox.askyesno(
        "Confirm Delete",
        f"Are you sure you want to delete {website}?"
    )

    if confirm:
        del passwords[website]
        rewrite_password_file()

        messagebox.showinfo(
            "Deleted",
            "Password deleted successfully!"
        )

        clear_fields()


def view_passwords():
    if not passwords:
        messagebox.showinfo(
            "Saved Passwords",
            "No passwords saved!"
        )
        return

    window = tk.Toplevel(dashboard)
    window.title("Saved Passwords")
    window.geometry("550x450")

    tk.Label(
        window,
        text="Saved Passwords",
        font=("Arial", 18, "bold")
    ).pack(pady=15)

    for website, password in passwords.items():
        frame = tk.Frame(window)
        frame.pack(fill="x", padx=20, pady=5)

        tk.Label(
            frame,
            text=website,
            width=20,
            anchor="w",
            font=("Arial", 11, "bold")
        ).pack(side="left")

        tk.Label(
            frame,
            text=password,
            width=25,
            anchor="w"
        ).pack(side="left")


def generate_password():
    characters = (
        string.ascii_letters
        + string.digits
        + "@#$%^&*"
    )

    password = "".join(
        random.choice(characters)
        for _ in range(12)
    )

    password_result.delete(0, tk.END)
    password_result.insert(0, password)


def copy_generated_password():
    password = password_result.get()

    if not password:
        messagebox.showwarning(
            "Warning",
            "Generate a password first!"
        )
        return

    dashboard.clipboard_clear()
    dashboard.clipboard_append(password)

    messagebox.showinfo(
        "Copied",
        "Password copied to clipboard!"
    )


def toggle_password():
    if new_password_entry.cget("show") == "*":
        new_password_entry.config(show="")
        show_button.config(text="Hide")
    else:
        new_password_entry.config(show="*")
        show_button.config(text="Show")


def clear_fields():
    website_entry.delete(0, tk.END)
    new_password_entry.delete(0, tk.END)


def open_dashboard():
    global dashboard
    global website_entry
    global new_password_entry
    global search_entry
    global result_label
    global password_result
    global show_button

    dashboard = tk.Tk()
    dashboard.title("SecurePass")
    dashboard.geometry("700x700")

    tk.Label(
        dashboard,
        text="SECUREPASS",
        font=("Arial", 26, "bold")
    ).pack(pady=20)

    tk.Label(
        dashboard,
        text="Secure Password Manager"
    ).pack()

    tk.Label(
        dashboard,
        text="Website",
        font=("Arial", 12, "bold")
    ).pack(pady=(25, 5))

    website_entry = tk.Entry(
        dashboard,
        width=45
    )
    website_entry.pack()

    tk.Label(
        dashboard,
        text="Password",
        font=("Arial", 12, "bold")
    ).pack(pady=(15, 5))

    password_frame = tk.Frame(dashboard)
    password_frame.pack()

    new_password_entry = tk.Entry(
        password_frame,
        width=35,
        show="*"
    )
    new_password_entry.pack(side="left")

    show_button = tk.Button(
        password_frame,
        text="Show",
        width=7,
        command=toggle_password
    )
    show_button.pack(side="left", padx=5)

    button_frame = tk.Frame(dashboard)
    button_frame.pack(pady=15)

    tk.Button(
        button_frame,
        text="Save",
        width=12,
        command=save_password
    ).grid(row=0, column=0, padx=5)

    tk.Button(
        button_frame,
        text="Update",
        width=12,
        command=update_password
    ).grid(row=0, column=1, padx=5)

    tk.Button(
        button_frame,
        text="Delete",
        width=12,
        command=delete_password
    ).grid(row=0, column=2, padx=5)

    tk.Button(
        dashboard,
        text="Clear",
        width=15,
        command=clear_fields
    ).pack(pady=5)

    tk.Button(
        dashboard,
        text="View All Passwords",
        width=25,
        command=view_passwords
    ).pack(pady=15)

    tk.Label(
        dashboard,
        text="Search Website",
        font=("Arial", 12, "bold")
    ).pack(pady=(15, 5))

    search_entry = tk.Entry(
        dashboard,
        width=45
    )
    search_entry.pack()

    tk.Button(
        dashboard,
        text="Search",
        width=15,
        command=search_password
    ).pack(pady=8)

    result_label = tk.Label(
        dashboard,
        text="",
        font=("Arial", 11)
    )
    result_label.pack()

    tk.Label(
        dashboard,
        text="Password Generator",
        font=("Arial", 14, "bold")
    ).pack(pady=(25, 8))

    password_result = tk.Entry(
        dashboard,
        width=45
    )
    password_result.pack()

    generator_frame = tk.Frame(dashboard)
    generator_frame.pack(pady=8)

    tk.Button(
        generator_frame,
        text="Generate",
        width=15,
        command=generate_password
    ).grid(row=0, column=0, padx=5)

    tk.Button(
        generator_frame,
        text="Copy",
        width=15,
        command=copy_generated_password
    ).grid(row=0, column=1, padx=5)

    tk.Button(
        dashboard,
        text="Exit",
        width=15,
        command=dashboard.destroy
    ).pack(pady=25)

    dashboard.mainloop()


def create_master_window():
    global login_window
    global password_entry
    global confirm_entry

    login_window.destroy()

    login_window = tk.Tk()
    login_window.title("Create Master Password")
    login_window.geometry("450x400")

    tk.Label(
        login_window,
        text="Create Master Password",
        font=("Arial", 20, "bold")
    ).pack(pady=30)

    tk.Label(
        login_window,
        text="Create password:"
    ).pack()

    password_entry = tk.Entry(
        login_window,
        width=30,
        show="*"
    )
    password_entry.pack(pady=8)

    tk.Label(
        login_window,
        text="Confirm password:"
    ).pack()

    confirm_entry = tk.Entry(
        login_window,
        width=30,
        show="*"
    )
    confirm_entry.pack(pady=8)

    tk.Button(
        login_window,
        text="Create",
        width=15,
        command=create_master_password
    ).pack(pady=25)

    login_window.mainloop()


def start_login():
    global login_window
    global password_entry

    login_window = tk.Tk()
    login_window.title("SecurePass Login")
    login_window.geometry("450x350")

    tk.Label(
        login_window,
        text="SECUREPASS",
        font=("Arial", 26, "bold")
    ).pack(pady=35)

    tk.Label(
        login_window,
        text="Master Password",
        font=("Arial", 12)
    ).pack()

    password_entry = tk.Entry(
        login_window,
        width=30,
        show="*"
    )
    password_entry.pack(pady=10)

    tk.Button(
        login_window,
        text="LOGIN",
        width=18,
        command=login
    ).pack(pady=25)

    login_window.mainloop()


start_login()